# U10_C1 — Reading a PDB File
**Course:** Concepts of Bioinformatics (CBBI-501) · **Unit 10:** Structure Databases
**Type:** Course notebook · **Time:** 40 min · **Difficulty:** ★★☆
**Goes with:** Lectures L26–L27, Practicals S12–S13 (PDB & VMD)
**Before this:** U09_C2

## 1. What you will learn
- **Navigate** the Structure-Model-Chain-Residue-Atom (SMCRA) object hierarchy in Biopython's `Bio.PDB` [LO7].
- **Differentiate** standard polymer amino acid records (`ATOM`) from non-polymer heteroatoms/ligands (`HETATM`) [LO7].
- **Extract** 3D Cartesian coordinates $(x, y, z)$ and compute Euclidean distances between functional atoms [LO7].
- **Locate** the haem-binding pocket and measure the coordination distance between proximal His92 and the iron atom [LO7].


## 2. Biological question
Human adult hemoglobin is a heterotetramer ($\alpha_2\beta_2$) containing four heme groups. Each heme group binds one molecule of oxygen ($O_2$) reversibly at an iron (Fe) atom coordinated by a conserved proximal histidine residue. By inspecting the 2.0 Å X-ray crystal structure of deoxyhemoglobin (PDB `1A3N`), how do we navigate chains A, B, C, D, extract 3D coordinates, and verify the physical distance between His92 and the iron atom?


In [ ]:
import sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "biopython"], check=True)

NOTEBOOK_ID = "U10_C1"
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").is_dir())
DATA_DIR = ROOT / "data"
OUT_DIR = ROOT / "outputs" / NOTEBOOK_ID
OUT_DIR.mkdir(parents=True, exist_ok=True)

import Bio
print("Python", sys.version.split()[0], "| Biopython", Bio.__version__)

## 3. Concept 1: The SMCRA Data Hierarchy
### Theory
Biopython structures macromolecular crystallographic models using the **SMCRA** object architecture:
- **`Structure`**: The entire entry (e.g. PDB ID `1A3N`).
- **`Model`**: An individual conformational state (X-ray structures typically contain 1 model, index 0; NMR entries contain an ensemble of 10–30 models).
- **`Chain`**: An individual polypeptide or nucleic acid macromolecule (e.g. Chain A, B, C, D in hemoglobin).
- **`Residue`**: An individual amino acid or hetero group. Identified by a tuple `(' ', res_number, insertion_code)`.
- **`Atom`**: An individual atomic center storing 3D coordinates $(x, y, z)$, B-factor (temperature factor), and occupancy.


In [ ]:
# Worked Example: Parsing 1A3N with Bio.PDB
from Bio.PDB import PDBParser
import numpy as np

parser = PDBParser(QUIET=True)
pdb_path = DATA_DIR / "structures" / "1a3n.pdb"
structure = parser.get_structure("1A3N", pdb_path)

model = structure[0]
chains = list(model.get_chains())
chain_ids = [c.id for c in chains]

print("=== PDB Structure: 1A3N (Human Deoxyhemoglobin) ===")
print("Header Title:", structure.header.get("name", "Deoxyhemoglobin"))
print("Resolution:  ", structure.header.get("resolution", 2.0), "Å")
print(f"Number of Chains in Model 0: {len(chains)} ({chain_ids})")


## 4. Concept 2: ATOM vs. HETATM Records
### Theory
The PDB flat file distinguishes standard biological residues from non-standard components:
- **`ATOM`**: Standard amino acid or nucleotide residues that form the covalently bonded polymer chain. In Biopython residue ID: `id[0] == ' '`.
- **`HETATM`**: Hetero-atoms, including prosthetic groups (haem `HEM`), cofactors, crystallographic waters (`HOH`), and small-molecule drugs. In Biopython: `id[0].startswith('H_')` or `'W'`.


In [ ]:
# Worked Example: Inspecting Chain B and finding the Haem Ligand
chain_b = model["B"]

amino_acids = [r for r in chain_b if r.id[0] == " "]
hetero_groups = [r for r in chain_b if r.id[0] != " "]

print(f"Chain B (Beta globin):")
print(f"  Standard amino acid residues: {len(amino_acids)}")
print(f"  Hetero residues (HETATM):     {len(hetero_groups)}")

# Locate the haem prosthetic group
haem_res = next(r for r in hetero_groups if r.resname == "HEM")
print("\nHaem Prosthetic Group Details:")
print("  Residue Name: ", haem_res.resname)
print("  Residue ID:   ", haem_res.id)
print("  Number of Atoms in Haem:", len(haem_res))

fe_atom = haem_res["FE"]
print(f"  Iron Atom Coordinates: x={fe_atom.coord[0]:.3f}, y={fe_atom.coord[1]:.3f}, z={fe_atom.coord[2]:.3f}")


## 5. Concept 3: Measuring Physical 3D Distances
### Theory
In structural biophysics, the distance between two atoms $A$ and $B$ with coordinates $(x_1, y_1, z_1)$ and $(x_2, y_2, z_2)$ is the 3D Euclidean distance:
$$d(A, B) = \sqrt{(x_2 - x_1)^2 + (y_2 - y_1)^2 + (z_2 - z_1)^2}$$
In Biopython, subtracting two `Atom` objects (`atomA - atomB`) automatically computes this Euclidean distance in Angstroms (Å)!


In [ ]:
# Worked Example: Measuring proximal His92 coordination distance
# Residue 92 in chain B is the proximal histidine
his92 = chain_b[92]
print("Residue 92 Name:", his92.resname)

# NE2 is the epsilon-nitrogen that coordinates the Fe atom
ne2_atom = his92["NE2"]
coord_distance = ne2_atom - fe_atom

print(f"His92 NE2 Coordinates: {ne2_atom.coord}")
print(f"Haem Fe Coordinates:   {fe_atom.coord}")
print(f"Physical Coordination Distance (NE2 - Fe): {coord_distance:.2f} Å")
print("Physiological covalent coordination bond distance is typically ~2.1 - 2.3 Å!")


## 6. Concept 4: Mapping the Haem Pocket
### Theory
Any residue with an atom within $4.5$ Å of the haem group constitutes part of the haem-binding pocket that stabilizes the porphyrin ring through hydrophobic packing.


In [ ]:
# Worked Example: Identifying residues in the haem pocket within 4.5 Å
pocket_residues = set()

for res in amino_acids:
    for atom_res in res:
        for atom_haem in haem_res:
            if (atom_res - atom_haem) < 4.5:
                pocket_residues.add((res.id[1], res.resname))
                break

print(f"Found {len(pocket_residues)} amino acids in Chain B within 4.5 Å of the haem group:")
for res_num, res_name in sorted(pocket_residues):
    print(f"  Residue {res_num:3}: {res_name}")


## 7. Summary
- `Bio.PDB.PDBParser` constructs a hierarchical SMCRA tree (Structure, Model, Chain, Residue, Atom).
- `ATOM` records define standard protein residues; `HETATM` records define ligands, cofactors, and waters.
- Subtracting two atoms (`atomA - atomB`) returns Euclidean distance in Angstroms.
- Proximal His92 directly coordinates the haem iron atom at $\approx 2.26$ Å in 1A3N.


## 8. Next Steps
- Analyze sickle-cell hemoglobin (2HBS) in `U10_P_PDB_Structures.ipynb`.
- Self-test structure database concepts in `U10_Q_Structure_Databases.ipynb`.
